In [ ]:
# Notebook 02: L2a Dummy Data Generation
# Produces L2a_daily.csv, L2a_daily.json, L2a_weekly.csv, L2a_weekly.json
 
# Cell 1: Imports and global constants
# All window, boundary, and date range settings defined here only

import pandas as pd
import numpy as np
import json
from scipy.stats import truncnorm

In [ ]:
# Cell 2: Date spine and daily L1-level totals
# Totals used as day denominators when distributing SF across categories

SEED       = 77
RNG        = np.random.default_rng(SEED)
WINDOW     = 7
HALF       = WINDOW // 2
START      = "2023-01-01"
END        = "2023-12-31"
DATA_START = "2023-01-04"
DATA_END   = "2023-12-28"
ZERO_DATES = ["2023-12-25"]
NULL_DATES = []
METRICS    = ["GHGE", "LU", "WU"]
 
print("Constants loaded")
print(f"Output date range: {DATA_START} to {DATA_END}")
 
dates        = pd.date_range(START, END, freq="D")
N            = len(dates)
day_idx      = np.arange(N)
 
def trunc(mean, std, low, high, size):
    a = (low  - mean) / std
    b = (high - mean) / std
    return truncnorm.rvs(a, b, loc=mean, scale=std, size=size, random_state=SEED)
 
seasonal     = 1.0 + 0.18 * np.sin((day_idx / N) * 2 * np.pi + np.pi * 0.3)
is_weekend   = pd.Series(dates).dt.dayofweek.isin([5, 6]).astype(float).values
weekend_bump = 1.0 + 0.08 * is_weekend
 
total_GHGE_SF = trunc(820_000,      95_000,     500_000,    1_100_000,    N) * seasonal * weekend_bump
total_LU_SF   = trunc(2_100_000,   480_000,     900_000,    3_400_000,    N) * seasonal * weekend_bump
total_WU_SF   = trunc(140_000_000, 28_000_000,  60_000_000, 200_000_000,  N) * seasonal * weekend_bump
 
print(f"Date spine: {N} days, {dates[0].date()} to {dates[-1].date()}")

In [ ]:
# Cell 3: LCFS category names - 65 categories, ONS LCFS classification
# Names match the original L2a dummy file exactly

LCFS_CATS = [
    "Apples", "Bacon and ham", "Baker's yeast, dessert preparations, soups",
    "Bananas", "Beef", "Berries", "Bread", "Buns, crispbread and biscuits",
    "Butter", "Cabbages", "Cakes and puddings", "Cheese and curd", "Chocolate",
    "Citrus fruits", "Cocoa and powdered chocolate", "Coffee",
    "Confectionery products", "Dairy alternative", "Dried fruit and nuts",
    "Dried vegetables", "Edible oils and other edible animal fats",
    "Edibles ices and ice cream", "Eggs", "Fish", "Fresh vegetables",
    "Fruit and vegetable juices", "Jams, marmalades", "Lamb",
    "Leaf and stem vegetables",
    "Margarine, other vegetable fats and peanut butter", "Meat alternative",
    "Milk", "Mineral or spring waters", "Offal, pate etc", "Olive oil",
    "Other breads and cereals", "Other fresh, chilled or frozen fruits",
    "Other fresh, chilled, or frozen edible meat",
    "Other milk products", "Other preserved or processed fish & seafood",
    "Other preserved or processed meat", "Other preserved or processed veg",
    "Other sugar products", "Pasta products", "Pastry (savoury)", "Pears",
    "Pork", "Potatoes", "Poultry", "Preserved fruits and fruit based products",
    "Preserved milk", "Ready meals", "Rice",
    "Root crops, non-starchy bulbs and mushrooms",
    "Salt, spices, herbs & other food products", "Sauces, condiments",
    "Sausages", "Savoury snacks", "Seafood, dried, smoked or salted fish",
    "Soft drinks (inc. ready to drink fruit drinks)", "Stone fruits", "Sugar",
    "Tea", "Vegetable grown for their fruit", "Yoghurt",
]
N_CATS = len(LCFS_CATS)
print(f"Categories: {N_CATS}  (expected 65)")

In [ ]:
# Cell 4: Generate daily L2a rows for all categories and dates
# Dirichlet shares distribute each day total independently per metric

rows = []
for i, d in enumerate(dates):
    date_str    = d.strftime("%Y-%m-%d")
    ghge_shares = RNG.dirichlet(np.full(N_CATS, 1.6))
    lu_shares   = RNG.dirichlet(np.full(N_CATS, 1.4))
    wu_shares   = RNG.dirichlet(np.full(N_CATS, 1.8))
 
    cat_GHGE = ghge_shares * total_GHGE_SF[i]
    cat_LU   = lu_shares   * total_LU_SF[i]
    cat_WU   = wu_shares   * total_WU_SF[i]
 
    cat_GHGE_pkg = np.clip(trunc(5.2,   1.1,  2.0,  9.0,  N_CATS) + RNG.normal(0, 1.0,  N_CATS), 0.5,  25.0)
    cat_LU_pkg   = np.clip(trunc(11.8,  2.6,  4.0,  22.0, N_CATS) + RNG.normal(0, 2.0,  N_CATS), 1.0,  50.0)
    cat_WU_pkg   = np.clip(trunc(270.0, 55.0, 80.0, 500.0, N_CATS) + RNG.normal(0, 35.0, N_CATS), 10.0, 800.0)
 
    ghge_ranks = pd.Series(cat_GHGE).rank(ascending=False, method="min").astype(int).values
    lu_ranks   = pd.Series(cat_LU).rank(ascending=False, method="min").astype(int).values
    wu_ranks   = pd.Series(cat_WU).rank(ascending=False, method="min").astype(int).values
 
    for j, cat in enumerate(LCFS_CATS):
        rows.append({
            "date"              : date_str,
            "lcfs_cat"          : cat,
            "total_GHGE_SF"     : round(float(cat_GHGE[j]),    4),
            "total_LU_SF"       : round(float(cat_LU[j]),      4),
            "total_WU_SF"       : round(float(cat_WU[j]),      4),
            "avg_GHGE_perkg"    : round(float(cat_GHGE_pkg[j]),6),
            "avg_LU_perkg"      : round(float(cat_LU_pkg[j]),  6),
            "avg_WU_perkg"      : round(float(cat_WU_pkg[j]),  6),
            "GHGE_rank_on_day"  : int(ghge_ranks[j]),
            "LU_rank_on_day"    : int(lu_ranks[j]),
            "WU_rank_on_day"    : int(wu_ranks[j]),
        })
 
L2a = pd.DataFrame(rows)
L2a = L2a.sort_values(["lcfs_cat", "date"]).reset_index(drop=True)
print(f"L2a shape: {L2a.shape}  (expected {N * N_CATS} rows)") 

In [ ]:
# Cell 5: Apply closure masks and sort
# Zero dates included as 0 in rolling windows; null dates excluded entirely

SF_COLS      = ["total_GHGE_SF", "total_LU_SF", "total_WU_SF"]
PKG_COLS     = ["avg_GHGE_perkg", "avg_LU_perkg", "avg_WU_perkg"]
ALL_VAL_COLS = SF_COLS + PKG_COLS
 
for d in ZERO_DATES:
    mask = L2a["date"] == d
    L2a.loc[mask, ALL_VAL_COLS] = 0.0
    print(f"Zero mask: {d} ({int(mask.sum())} rows)")
 
for d in NULL_DATES:
    mask = L2a["date"] == d
    L2a.loc[mask, ALL_VAL_COLS] = np.nan
    print(f"Null mask: {d} ({int(mask.sum())} rows)")

In [ ]:
# Cell 6: Per-category 7-day rolling means for all SF and per-kg columns
# Rolling applied before boundary masking - same logic as L1 notebook

def rolling_clean_cat(series):
    # Step 1: rolling mean with strict full-window requirement
    rolled = series.rolling(window=WINDOW, center=True, min_periods=WINDOW).mean()
    # Step 2: mask boundary rows after rolling
    rolled.iloc[:HALF]  = np.nan
    rolled.iloc[-HALF:] = np.nan
    return rolled
 
for m, src in zip(METRICS, SF_COLS):
    L2a[f"roll7_{m}_SF"] = (
        L2a.groupby("lcfs_cat")[src]
           .transform(rolling_clean_cat)
           .round(6)
    )
 
for m, src in zip(METRICS, PKG_COLS):
    L2a[f"roll7_{m}_perkg"] = (
        L2a.groupby("lcfs_cat")[src]
           .transform(rolling_clean_cat)
           .round(6)
    )
 
print("Rolling column verification (all must show n=359 per category):")
roll_cols = [f"roll7_{m}_SF" for m in METRICS] + [f"roll7_{m}_perkg" for m in METRICS]
for col in roll_cols:
    counts = L2a.groupby("lcfs_cat")[col].apply(lambda s: s.notna().sum())
    first  = L2a.loc[L2a[col].notna(), "date"].min()
    last   = L2a.loc[L2a[col].notna(), "date"].max()
    flag   = "OK" if counts.min() == 359 and counts.max() == 359 else "MISMATCH"
    print(f"  {col:<25} n={counts.min()}-{counts.max()}  {first} -> {last}  {flag}")

In [ ]:
# Cell 7: Annual statistics per category from smoothed columns
# All reference values derived from smoothed series for consistency with charts
 
for m in METRICS:
    sf_roll  = f"roll7_{m}_SF"
    pkg_roll = f"roll7_{m}_perkg"
 
    cat_sf_mean  = L2a.groupby("lcfs_cat")[sf_roll].transform("mean")
    cat_pkg_mean = L2a.groupby("lcfs_cat")[pkg_roll].transform("mean")
    cat_pkg_min  = L2a.groupby("lcfs_cat")[pkg_roll].transform("min")
    cat_pkg_max  = L2a.groupby("lcfs_cat")[pkg_roll].transform("max")
 
    L2a[f"{m}_annual_mean"]    = cat_sf_mean.round(6)
    L2a[f"{m}_annual_min"]     = L2a.groupby("lcfs_cat")[sf_roll].transform("min").round(6)
    L2a[f"{m}_annual_max"]     = L2a.groupby("lcfs_cat")[sf_roll].transform("max").round(6)
 
    # Annual share: category smoothed total as fraction of all-category smoothed total
    grand_total               = L2a[sf_roll].sum()
    cat_total                 = L2a.groupby("lcfs_cat")[sf_roll].transform("sum")
    L2a[f"{m}_annual_share"]  = ((cat_total / grand_total) * 100).round(6)
 
    # Annual rank by smoothed annual mean: 1 = highest impact category
    sf_means                  = L2a.groupby("lcfs_cat")[sf_roll].mean()
    L2a[f"{m}_annual_rank"]   = L2a["lcfs_cat"].map(
        sf_means.rank(ascending=False, method="min").astype(int)
    )
 
    # Pct change: smoothed daily value vs smoothed annual mean for that category
    L2a[f"{m}_pct_from_annual_mean"] = (
        ((L2a[sf_roll] - cat_sf_mean) / cat_sf_mean) * 100
    ).round(6)
 
    L2a[f"{m}_perkg_annual_mean"] = cat_pkg_mean.round(6)
    L2a[f"{m}_perkg_annual_min"]  = cat_pkg_min.round(6)
    L2a[f"{m}_perkg_annual_max"]  = cat_pkg_max.round(6)
 
    # Max absolute deviation: symmetric half-width for diverging colour scale in D3
    L2a[f"{m}_perkg_max_abs_dev"] = np.maximum(
        (cat_pkg_max - cat_pkg_mean).abs(),
        (cat_pkg_mean - cat_pkg_min).abs()
    ).round(6)
 
    pkg_means                         = L2a.groupby("lcfs_cat")[pkg_roll].mean()
    L2a[f"{m}_perkg_annual_rank"]     = L2a["lcfs_cat"].map(
        pkg_means.rank(ascending=False, method="min").astype(int)
    )
 
    L2a[f"{m}_perkg_pct_from_annual_mean"] = (
        ((L2a[pkg_roll] - cat_pkg_mean) / cat_pkg_mean) * 100
    ).round(6)
 
print(f"Annual stats complete - L2a columns: {L2a.shape[1]}")

In [ ]:
# Cell 8: Weekly aggregation from smoothed values
# Filtered to DATA_START-DATA_END first so all roll7 inputs are non-null

def get_week_start(date_str):
    # Jan-01 anchored 7-day bins regardless of day of week.
    # Week 1: Jan-01 to Jan-07, Week 2: Jan-08 to Jan-14, etc.
    d    = pd.Timestamp(date_str)
    jan1 = pd.Timestamp(d.year, 1, 1)
    bin_num    = (d - jan1).days // 7
    week_start = jan1 + pd.Timedelta(days=7 * bin_num)
    return week_start.strftime("%Y-%m-%d")

# Assign Jan-01-anchored week_start for all L2a rows
L2a["week_start"] = L2a["date"].apply(get_week_start)
 
# Majority calendar month per week (0-indexed: Jan=0, Dec=11)
week_month_map = (
    L2a.groupby("week_start")["date"]
    .apply(lambda s: (pd.to_datetime(s).dt.month - 1).value_counts().idxmax())
    .reset_index()
    .rename(columns={"date": "calendar_month"})
)
 
# Filter to valid smoothed range before groupby - ensures no NaN weekly means
L2a_smooth = L2a[
    (L2a["date"] >= DATA_START) &
    (L2a["date"] <= DATA_END)
].copy()
 
grp      = L2a_smooth.groupby(["week_start", "lcfs_cat"])
agg_dict = {}
for m in METRICS:
    agg_dict[f"{m}_weekly_mean"]       = (f"roll7_{m}_SF",    "mean")
    agg_dict[f"{m}_perkg_weekly_mean"] = (f"roll7_{m}_perkg", "mean")
 
weekly = grp.agg(**{k: v for k, v in agg_dict.items()}).reset_index()
weekly = weekly.sort_values(["week_start", "lcfs_cat"]).reset_index(drop=True)
 
# Weekly ranks computed only on filtered weeks - no NaN means so astype(int) is safe
for m in METRICS:
    weekly[f"{m}_weekly_rank"] = (
        weekly.groupby("week_start")[f"{m}_weekly_mean"]
        .rank(ascending=False, method="min")
        .astype(int)
    )
 
# Merge calendar month and all annual reference columns
annual_ref_cols = (
    ["lcfs_cat"]
    + [f"{m}_{s}" for m in METRICS for s in [
        "annual_mean", "annual_min", "annual_max", "annual_share", "annual_rank",
        "perkg_annual_mean", "perkg_annual_min", "perkg_annual_max",
        "perkg_max_abs_dev", "perkg_annual_rank",
    ]]
)
annual_ref = (
    L2a[annual_ref_cols]
    .drop_duplicates("lcfs_cat")
    .reset_index(drop=True)
)
 
weekly = weekly.merge(annual_ref,       on="lcfs_cat",   how="left")
weekly = weekly.merge(week_month_map,   on="week_start", how="left")
weekly["calendar_month"] = weekly["calendar_month"].astype(int)
 
for m in METRICS:
    weekly[f"{m}_pct_from_annual_mean"] = (
        ((weekly[f"{m}_weekly_mean"] - weekly[f"{m}_annual_mean"])
         / weekly[f"{m}_annual_mean"]) * 100
    ).round(6)
    weekly[f"{m}_perkg_pct_from_annual_mean"] = (
        ((weekly[f"{m}_perkg_weekly_mean"] - weekly[f"{m}_perkg_annual_mean"])
         / weekly[f"{m}_perkg_annual_mean"]) * 100
    ).round(6)
 
for col in [c for c in weekly.columns if "mean" in c.lower()]:
    weekly[col] = weekly[col].round(6)
 
print(f"Weekly shape: {weekly.shape}")
print(f"Weeks: {weekly['week_start'].nunique()}, Categories: {weekly['lcfs_cat'].nunique()}")

In [ ]:
# Cell 9: Define export column lists and filter to valid date range
# Daily filtered to DATA_START-DATA_END; weekly filtered to week_start >= DATA_START

DAILY_COLS = (
    ["date", "lcfs_cat"]
    + [f"roll7_{m}_SF"    for m in METRICS]
    + [f"roll7_{m}_perkg" for m in METRICS]
    + [f"{m}_rank_on_day" for m in METRICS]
    + [f"{m}_{s}" for m in METRICS for s in [
        "annual_mean", "annual_min", "annual_max", "annual_share", "annual_rank",
        "pct_from_annual_mean",
        "perkg_annual_mean", "perkg_annual_min", "perkg_annual_max",
        "perkg_max_abs_dev", "perkg_annual_rank", "perkg_pct_from_annual_mean",
    ]]
)
 
WEEKLY_COLS = (
    ["week_start", "lcfs_cat", "calendar_month"]
    + [f"{m}_weekly_mean"       for m in METRICS]
    + [f"{m}_perkg_weekly_mean" for m in METRICS]
    + [f"{m}_weekly_rank"       for m in METRICS]
    + [f"{m}_{s}" for m in METRICS for s in [
        "annual_mean", "annual_min", "annual_max", "annual_share", "annual_rank",
        "pct_from_annual_mean",
        "perkg_annual_mean", "perkg_annual_min", "perkg_annual_max",
        "perkg_max_abs_dev", "perkg_annual_rank", "perkg_pct_from_annual_mean",
    ]]
)

# Full year export: Jan-01 to Dec-31
# Rolling columns are null for boundary dates - chart skips null cells naturally
L2a_daily  = L2a[DAILY_COLS].copy().reset_index(drop=True)
L2a_weekly = weekly[WEEKLY_COLS].copy().reset_index(drop=True)
 
# Check for missing columns
for label, df, cols in [("DAILY", L2a, DAILY_COLS), ("WEEKLY", weekly, WEEKLY_COLS)]:
    missing = [c for c in cols if c not in df.columns]
    print(f"{label} missing: {missing if missing else 'None'}")
 
expected_daily = (pd.to_datetime(DATA_END) - pd.to_datetime(DATA_START)).days + 1
print(f"Daily rows:  {len(L2a_daily)} (expected {expected_daily} dates x {N_CATS} cats)")
print(f"Weekly rows: {len(L2a_weekly)} (weeks: {L2a_weekly['week_start'].nunique()}, cats: {L2a_weekly['lcfs_cat'].nunique()})")

In [ ]:
# Cell 10: Export CSV and JSON for daily and weekly files with spot checks
# Verifies date ranges, null counts, and absence of the Dec-26-2022 edge week
 
# L2a_daily.to_csv("./data/L2a_daily.csv",   index=False)
# L2a_weekly.to_csv("./data/L2a_weekly.csv", index=False)
# print("Saved: L2a_daily.csv and L2a_weekly.csv")
 
L2a_daily.to_json("./data/L2a_daily.json",   orient="records", indent=2)
L2a_weekly.to_json("./data/L2a_weekly.json", orient="records", indent=2)
print("Saved: L2a_daily.json and L2a_weekly.json")
 
with open("./data/L2a_daily.json")  as f: d_check = json.load(f)
with open("./data/L2a_weekly.json") as f: w_check = json.load(f)
 
d_dates = sorted(set(r["date"] for r in d_check))
w_weeks = sorted(set(r["week_start"] for r in w_check))
print(f"Daily  date range: {d_dates[0]} -> {d_dates[-1]}")
print(f"Weekly week range: {w_weeks[0]} -> {w_weeks[-1]}")
 
# Null check on all rolling columns
print("Null check on roll7 columns in daily JSON (all must be 0):")
for col in [f"roll7_{m}_SF" for m in METRICS] + [f"roll7_{m}_perkg" for m in METRICS]:
    nulls = sum(1 for r in d_check if r.get(col) is None)
    print(f"  {col:<25} {'OK' if nulls == 0 else 'NULLS: ' + str(nulls)}")
 
# Confirm Dec-26-2022 week is absent
dec26 = [r for r in w_check if r["week_start"] == "2022-12-26"]
print(f"Week 2022-12-26 in weekly JSON: {'FOUND - CHECK' if dec26 else 'absent OK'}")
 
# Sample row
sample = next(r for r in d_check if r["roll7_GHGE_SF"] is not None)
print(f"Sample daily row ({sample['date']}, {sample['lcfs_cat']}):")
for col in ["roll7_GHGE_SF", "roll7_GHGE_perkg",
            "GHGE_annual_mean", "GHGE_perkg_annual_mean",
            "GHGE_annual_share", "GHGE_perkg_max_abs_dev",
            "GHGE_pct_from_annual_mean"]:
    print(f"  {col:<35} {sample.get(col)}")
 
print("All exports complete")